# 02 Pipeline

Build a **governed, quality-checked, Microsoft Fabric notebook workflow** with independently cloneable indexed source and target blocks: **0. Environment → E. Extract → T. Transform → L. Load**.

## Tested with FabricOps

The previous baseline was run in Microsoft Fabric with FabricOps v0.2.0 by Voyce on 6 Aug 2026. This redesigned workflow has local structural and public-API compatibility validation only; run it in your configured Fabric workspace before treating it as runtime-validated.

# 0. Environment

Run shared configuration, import public APIs, and initialize indexed state exactly once.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_schema,
    profile_and_register_table,
    profile_dataframe,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_pipeline_prep,
    read_warehouse_table,
    write_lakehouse_table,
    write_pipeline_prep,
    write_warehouse_table,
    widget_select_data_contract,
    widget_view_catalogue,
)

SOURCES = {}
SOURCE_PREPS = {}
SOURCE_DFS = {}
SOURCE_PROFILES = {}
SOURCE_RESULTS = {}
TARGETS = {}
TARGET_DFS = {}
TARGET_PREPS = {}
TARGET_RESULTS = {}
TARGET_CONTRACTS = {}

# E. Extract

The complete SOURCE 1 sequence is source-owned. Duplicate the complete block for another registered source.

## SOURCE 1 — Select

Select one registered table. Its Catalogue `table_id` is authoritative; physical coordinates are retained only for explicit IO.

In [ ]:
SOURCE = 1
source_catalogue = widget_view_catalogue(
    mode="explore",
    spark_session=spark,
)

## SOURCE 1 — Configure

Store the selected registered identity and its engineer-authored read strategy under the integer index.

In [ ]:
source_selection = source_catalogue["get_selection"]()
if not source_selection.get("table_id"):
    raise ValueError("Select a registered source table from the Catalogue first.")

SOURCES[SOURCE] = {
    "table_id": source_selection["table_id"],
    "store_type": source_selection["store_type"],
    "target": source_selection["layer"],
    "schema": source_selection.get("schema_name"),
    "table_name": source_selection["table_name"],
    "read_strategy": "incremental_watermark",
    "watermark_column": "modified_datetime",
    "partition_column": None,
}
source = SOURCES[SOURCE]

# Other valid read strategies:
# source["read_strategy"] = "full_dataset"
# source["watermark_column"] = None
# source["read_strategy"] = "incremental_partition"
# source["partition_column"] = "snapshot_date"

## SOURCE 1 — Prepare / Guard

Preparation resolves the registered physical source, source change/checkpoint state, runtime read mode, physical scope, and candidate completion state. No publication selection is needed.

In [ ]:
SOURCE_PREPS[SOURCE] = read_pipeline_prep(
    source_table_id=source["table_id"],
    source_read_strategy=source["read_strategy"],
    source_watermark_column=source["watermark_column"],
    source_partition_column=source["partition_column"],
)
source_prep = SOURCE_PREPS[SOURCE]

SOURCE_RESULTS[SOURCE] = [check_schema(table_id=source["table_id"])]
if source_prep["observation"] is not None:
    SOURCE_RESULTS[SOURCE].append(
        check_freshness(source_prep["observation"], table_id=source["table_id"])
    )
if source_prep["changes"] is not None:
    SOURCE_RESULTS[SOURCE].append(source_prep["changes"])
if not all(result["can_continue"] for result in SOURCE_RESULTS[SOURCE]):
    raise RuntimeError(f"A SOURCE {SOURCE} Guardrail blocked this run.")

SHOULD_RUN = source_prep["read_mode"] != "skip"
print(f'Runtime read mode: {source_prep["read_mode"]}')

## SOURCE 1 — Read

Dispatch the physical reader from Catalogue metadata and pass only the runtime processing scope.

In [ ]:
if not SHOULD_RUN:
    SOURCE_DFS[SOURCE] = None
    print("No source changes detected. Read and downstream publication are skipped.")
elif source["store_type"] == "warehouse":
    SOURCE_DFS[SOURCE] = read_warehouse_table(
        source["schema"],
        source["table_name"],
        target=source["target"],
        spark_session=spark,
        processing_scope=source_prep["scope"],
    )
elif source["store_type"] == "lakehouse":
    SOURCE_DFS[SOURCE] = read_lakehouse_table(
        source["table_name"],
        target=source["target"],
        schema=source["schema"],
        spark_session=spark,
        processing_scope=source_prep["scope"],
    )
else:
    raise ValueError(f'Unsupported registered source store_type: {source["store_type"]!r}')

## SOURCE 1 — DQ / Profile

DQ uses canonical source identity. Only a complete physical read may replace the registered profile; an incremental subset is diagnostic.

In [ ]:
if SHOULD_RUN:
    source_dq = check_dq(SOURCE_DFS[SOURCE], table_id=source["table_id"])
    SOURCE_RESULTS[SOURCE].append(source_dq)
    display(source_dq["summary"])
    if not source_dq["can_continue"]:
        raise RuntimeError(f"A SOURCE {SOURCE} DQ Guardrail blocked this run.")

    if source_prep["read_mode"] == "full_dataset":
        SOURCE_PROFILES[SOURCE] = profile_and_register_table(
            SOURCE_DFS[SOURCE],
            profile_role="source",
            target=source["target"],
            schema=source["schema"],
            table_name=source["table_name"],
        )
    elif source_prep["read_mode"] == "incremental_subset":
        SOURCE_PROFILES[SOURCE] = profile_dataframe(SOURCE_DFS[SOURCE])
    display(SOURCE_PROFILES[SOURCE])

### Optional raw file source

Raw files are not Catalogue table identities. Read a file as a full dataset, transform it, then publish to a governed registered table. Do not manufacture a `table_id` or apply registered-table change semantics to a file path.

In [ ]:
# Optional alternatives — choose one instead of the registered-table sequence above.
# SOURCE = 1
# SOURCE_DFS[SOURCE] = read_lakehouse_csv(
#     "Files/inbound/orders.csv", target="source", spark_session=spark,
#     header=True, inferSchema=True,
# )
# SOURCE_DFS[SOURCE] = read_lakehouse_excel(
#     "Files/inbound/orders.xlsx", target="source", spark_session=spark,
#     sheet_name="orders",
# )
# SOURCE_DFS[SOURCE] = read_lakehouse_parquet(
#     "Files/inbound/orders.parquet", target="source", spark_session=spark,
# )

### Need another registered source?

Duplicate the complete SOURCE block and change `SOURCE = 2`. Keep all state in the indexed dictionaries; do not create numbered variable names.

# T. Transform

**Business transformation is engineer-owned.** Keep joins, filters, derivations, aggregation, and reshaping visible.

In [ ]:
if SHOULD_RUN:
    transformed_df = SOURCE_DFS[1]
    display(transformed_df)

# After cloning the complete source block with SOURCE = 2:
# transformed_df = SOURCE_DFS[1].join(
#     SOURCE_DFS[2],
#     on="student_id",
#     how="left",
# )

# L. Load

Each complete TARGET block owns selection, Data Contract choice, Guardrails, preparation, and explicit physical publication.

## TARGET 1 — Select

Select the registered governed table only after transformation.

In [ ]:
TARGET = 1
target_catalogue = widget_view_catalogue(
    mode="explore",
    spark_session=spark,
)
target_selection = target_catalogue["get_selection"]()
if not target_selection.get("table_id"):
    raise ValueError("Select a registered target table from the Catalogue first.")

TARGETS[TARGET] = {
    "table_id": target_selection["table_id"],
    "store_type": target_selection["store_type"],
    "target": target_selection["layer"],
    "schema": target_selection.get("schema_name"),
    "table_name": target_selection["table_name"],
}
target = TARGETS[TARGET]

## TARGET 1 — Data Contract

Development uses current mutable authoring or the exact frozen selected contract. Production resolves the one active Data Contract.

In [ ]:
TARGET_CONTRACTS[TARGET] = widget_select_data_contract(
    table_id=target["table_id"]
)

## TARGET 1 — Guard

Assign the transformed DataFrame explicitly, then enforce target-owned Guardrails by canonical identity.

In [ ]:
if SHOULD_RUN:
    TARGET_DFS[1] = transformed_df
    target_df = TARGET_DFS[TARGET]
    target_schema_result = check_schema(
        table_id=target["table_id"],
        dataframe=target_df,
    )
    target_dq_result = check_dq(
        target_df,
        table_id=target["table_id"],
    )
    TARGET_RESULTS[TARGET] = [target_schema_result, target_dq_result]
    display(target_dq_result["summary"])
    if not all(result["can_continue"] for result in TARGET_RESULTS[TARGET]):
        raise RuntimeError(f"A TARGET {TARGET} Guardrail blocked publication.")

## TARGET 1 — Prepare

Target preparation resolves the selected table’s Data Contract, processing/load strategy, physical store, and source completion context.

In [ ]:
if SHOULD_RUN:
    TARGET_PREPS[TARGET] = write_pipeline_prep(
        target_df,
        target_table_id=target["table_id"],
        source_preps=[SOURCE_PREPS[1]],
    )
    target_prep = TARGET_PREPS[TARGET]

## TARGET 1 — Publish

The physical writer remains explicit. Candidate source progress is committed only after this target write succeeds.

In [ ]:
if SHOULD_RUN:
    if target["store_type"] == "lakehouse":
        TARGET_RESULTS[TARGET].append(write_lakehouse_table(
            target_prep["df"],
            target["table_name"],
            target=target["target"],
            schema=target["schema"],
            mode=target_prep["mode"],
            options=target_prep["options"],
            load_strategy=target_prep["load_strategy"],
            load_strategy_parameters=target_prep["load_strategy_parameters"],
            processing_scope=target_prep["scope"],
            completion_context=target_prep["completion"],
        ))
    elif target["store_type"] == "warehouse":
        TARGET_RESULTS[TARGET].append(write_warehouse_table(
            target_prep["df"],
            target["schema"],
            target["table_name"],
            target=target["target"],
            mode=target_prep["mode"],
            options=target_prep["options"],
            load_strategy=target_prep["load_strategy"],
            load_strategy_parameters=target_prep["load_strategy_parameters"],
            completion_context=target_prep["completion"],
        ))
    else:
        raise ValueError(f'Unsupported registered target store_type: {target["store_type"]!r}')

### Need another target?

FabricOps recommends one governed target per pipeline because target writes are independently committed and are not automatically rolled back together. Multiple targets are supported when needed by cloning the complete TARGET block and incrementing `TARGET`. Preserve canonical one-target completion ownership; do not treat cloned publications as atomic.

```python
# TARGET = 2
# TARGET_DFS[2] = summary_df
```